# Baseline RAG - Colab Runner
Thin runner notebook. Core logic stays in the Python modules.

## Clone private repo
Use a GitHub Personal Access Token with read access to this repo.

In [2]:
import getpass
import os
import subprocess

os.chdir('/content')

token = getpass.getpass('GitHub token (repo read access): ')
result = subprocess.run(
    ['git', 'clone', f'https://{token}@github.com/zahiTouqan/latent-rag.git'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Strip token from error output before printing
    print('STDOUT:', result.stdout.replace(token, '***'))
    print('STDERR:', result.stderr.replace(token, '***'))
    raise SystemExit('Clone failed')
else:
    print('Clone successful')

GitHub token (repo read access): ··········
Clone successful


In [3]:
%cd /content/latent-rag
!git fetch
!git checkout v2
!git pull origin v2
!python -m pip install -U pip
!pip install -r requirements.txt
!pip install "datasets>=2.14,<3.0"
!pip install -q --upgrade transformers

/content/latent-rag
Branch 'v2' set up to track remote branch 'v2' from 'origin'.
Switched to a new branch 'v2'
From https://github.com/zahiTouqan/latent-rag
 * branch            v2         -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.2 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [datasets]
ERROR: pip's dependency

In [24]:
!git pull origin v2


remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 460 bytes | 460.00 KiB/s, done.
From https://github.com/zahiTouqan/latent-rag
 * branch            v2         -> FETCH_HEAD
   3a53849..dd39e6b  v2         -> origin/v2
Updating 3a53849..dd39e6b
Fast-forward
 pipeline.py | 6 +++++-
 1 file changed, 5 insertions(+), 1 deletion(-)


## Download Some NQ data
Streams `facebook/dpr` NQs + related data so that we do not download all the questions on Colab. (Can be changed by modifying the number inside the square bracket)

In [4]:
import json, gzip, urllib.request

url = "https://dl.fbaipublicfiles.com/dpr/data/retriever/biencoder-nq-dev.json.gz"
with urllib.request.urlopen(url) as response:
    with gzip.open(response, 'rt', encoding='utf-8') as f:
        data = json.load(f)

sample = data[:100]

Just an example to visualize what an individual sample looks like

In [5]:
print("First training sample:")
import json
print(json.dumps(sample[0], indent=2))

First training sample:
{
  "dataset": "nq_dev_psgs_w100",
  "question": "who sings does he love me with reba",
  "answers": [
    "Linda Davis"
  ],
  "positive_ctxs": [
    {
      "title": "Does He Love You",
      "text": "Does He Love You \"Does He Love You\" is a song written by Sandy Knox and Billy Stritch, and recorded as a duet by American country music artists Reba McEntire and Linda Davis. It was released in August 1993 as the first single from Reba's album \"Greatest Hits Volume Two\". It is one of country music's several songs about a love triangle. \"Does He Love You\" was written in 1982 by Billy Stritch. He recorded it with a trio in which he performed at the time, because he wanted a song that could be sung by the other two members",
      "score": 1000,
      "title_score": 1,
      "passage_id": "11828866"
    },
    {
      "title": "Does He Love You",
      "text": "Does He Love You \"Does He Love You\" is a song written by Sandy Knox and Billy Stritch, and recorded

## Build mini Corpus based on the NQs
Each DPR sample has pre-chunked `positive_ctxs` (i.e. related documents) and `hard_negative_ctxs` (unrelated documents). We collect them from the data and build a mini corpus that we embed later (so that we do not download all documents on Colab :D)

In [6]:
import json
from pathlib import Path

CORPUS_OUT = Path("data/passages.jsonl")
CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)

corpus = {}  # passage_id -> record

for item in sample:
    for p in item["positive_ctxs"] + item["hard_negative_ctxs"]:
        pid = p["passage_id"]
        if pid not in corpus:
            corpus[pid] = {
                "id": pid,
                "text": f"{p['title']}: {p['text']}",
                "doc_id": p["title"]   # article title as document-level grouping
            }

with CORPUS_OUT.open("w", encoding="utf-8") as f:
    for record in corpus.values():
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(corpus)} passages to {CORPUS_OUT}")

Saved 10010 passages to data/passages.jsonl


## Build the index
This creates `index.faiss`, `metadata.jsonl`, and `config.json`. Make sure this step returns True

In [7]:
import torch
print(torch.cuda.is_available())  # should be True on T4

True


In [10]:
from huggingface_hub import login
from google.colab import userdata

# Safely get your token from the side panel
token = userdata.get('HF_TOKEN')

# Log HuggingFace library into your account
login(token)


Now we create the FAISS index of the corpus (should not take >4 mins for 10k passages)

In [13]:
INDEX_DIR = '/content/index'
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {INDEX_DIR}


Loaded 10010 passages from data/passages.jsonl
Loading embedding model (Encoder only): google/t5gemma-2-270m-270m
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 911/911 [00:00<00:00, 20374.49it/s]
Embedding 10010 passages (157 batches)...
  batch 10/157
  batch 20/157
  batch 30/157
  batch 40/157
  batch 50/157
  batch 60/157
  batch 70/157
  batch 80/157
  batch 90/157
  batch 100/157
  batch 110/157
  batch 120/157
  batch 130/157
  batch 140/157
  batch 150/157
  batch 157/157
Saving Latent Sequences to Safetensors...
Saved index to /content/index


In [14]:
!ls -lh /content/index


total 1.8G
-rw-r--r-- 1 root root  202 Apr 26 20:40 config.json
-rw-r--r-- 1 root root  25M Apr 26 20:40 index.faiss
-rw-r--r-- 1 root root 1.8G Apr 26 20:40 latents.safetensors
-rw-r--r-- 1 root root 6.8M Apr 26 20:40 metadata.jsonl


## Evaluate
We first need to build the eval file to compare the results to the ground truth. We build it from the DPR data. It is just a JSONL file with question, answer, and optionally relevant_ids fields.

In [20]:
import json
from pathlib import Path

EVAL_OUT = Path("data/eval.jsonl")

with EVAL_OUT.open("w", encoding="utf-8") as f:
    for item in sample:
        record = {
            "question": item["question"],
            "answer": item["answers"],
            "relevant_ids": [p["title"] for p in item["positive_ctxs"]]
        }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(sample)} eval examples to {EVAL_OUT}")

Saved 100 eval examples to data/eval.jsonl


Then we run the actual evaluation:

In [25]:
!python3 evaluate.py \
    --index_dir /content/index \
    --eval_path data/eval.jsonl \
    --retrieval_id_field source_doc_id \
    --top_k 5


Loaded 100 samples from data/eval.jsonl
Loading embedding model (Encoder only): google/t5gemma-2-270m-270m
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 911/911 [00:00<00:00, 5312.21it/s]
Loading generator model (Seq2Seq Full): google/t5gemma-2-270m-270m
Loading weights: 100% 911/911 [00:00<00:00, 20112.60it/s]
Loaded index with 10010 passages from data/passages.jsonl
100% 100/100 [11:38<00:00,  6.98s/it]

=== Results ===
  em                                  0.0000
  f1                                  0.0025
  generated_tokens                    112.5900
  generation_time_s                   6.8544
  latency_p50_ms                      7857.4278
  latency_p95_ms                      8299.0299
  recall@5                            0.0773
  retrieval_time_s                    0.0760
  total_time_s                        6.9796

Saved to results/results_eval_20260426_210551.json


In [ ]:
!cat results/results_eval_20260422_233310.json

{
  "config": {
    "eval_path": "data/eval.jsonl",
    "retrieval_id_field": "source_doc_id",
    "max_samples": null,
    "seed": 42,
    "index_dir": "/content/index",
    "top_k": 5,
    "max_new_tokens": 128,
    "generator_model": "Qwen/Qwen3-0.6B",
    "index": {
      "embedding_model": "BAAI/bge-base-en-v1.5",
      "corpus_path": "data/passages.jsonl",
      "passage_count": 10010,
      "max_docs": null,
      "doc_id_provided_count": 10010,
      "doc_id_missing_count": 0
    },
    "bertscore_enabled": false
  },
  "metrics": {
    "em": 0.0,
    "f1": 0.0317770989335791,
    "retrieval_time_s": 0.020343089030056945,
    "generation_time_s": 6.331013461430057,
    "total_time_s": 6.355677771040036,
    "generated_tokens": 128.0,
    "latency_p50_ms": 6374.689623500217,
    "latency_p95_ms": 6743.653761250334,
    "recall@5": 0.6355791800900829
  },
  "examples": [
    {
      "query": "who sings does he love me with reba",
      "gold_answers": [
        "Linda Davis"
    

Notes: as you can see in the output above, Qwen is repeating itself (probably because we need to use a proper chat template prompt or switch to a different model optimized for QA like flan T5), we can also try reducing the max_tokens if needed. The repeated keys in relevant_ids is not an issue as we convert it to a set in metrics.py